## Mapping UMLS vaccine terms to VO

In [1]:
import requests
from typing import List, Sequence, Tuple, Optional, Callable, Union
import pandas as pd
import pickle
from tqdm import tqdm
import concurrent.futures
import time
from threading import Lock
from functools import wraps

Rationale: UMLS has two concepts (CUIs) related to vaccines: 
1. Vaccines (C0042210)
2. Vaccine [APC] (C5399710)

The goal would be to: 
1. Base CUI [X]
2. Get all atoms (AUIs) of the two Vaccine concepts (occurences within source vocabularies) - Limited to english language. This yields 36 AUIs
3. Get descendents for all atoms (AUIs) extracted in previous step
4. For all unique concepts (CUIs) extracted in previous step, extract related atoms
5. For each newly identified CUI, extract relations again (with a depth limit) to find nested vaccine products/formulations

### UMLS API DOC [https://documentation.uts.nlm.nih.gov/rest/home.html]

In [2]:
apikey = '0c4d7175-6688-4c45-9a13-4c64d7354673'
uri = 'https://uts-ws.nlm.nih.gov/rest'

# Get descendants of vaccine concepts: 

In [ ]:
# !! get comparison of vocabs in umls vs athena !! 
 # BASIC STATS (vocab,  # concepts)

In [3]:
class RateLimiter:
    def __init__(self, max_calls_per_second=5):
        self.max_calls_per_second = max_calls_per_second
        self.min_interval = 1.0 / max_calls_per_second
        self.last_called = 0
        self.lock = Lock()
    
    def wait(self):
        with self.lock:
            elapsed = time.time() - self.last_called
            left_to_wait = self.min_interval - elapsed
            if left_to_wait > 0:
                time.sleep(left_to_wait)
            self.last_called = time.time()

rate_limiter = RateLimiter(max_calls_per_second=10)

In [4]:
def rate_limited(rate_limiter: RateLimiter):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            rate_limiter.wait()
            return func(*args, **kwargs)
        return wrapper
    return decorator

In [5]:
def threaded_umls_fetch(
    cui_list: List[str],
    apikey: str,
    fetch_func: Callable[[str, str], Union[dict, List[dict]]],
    desc: str = "Fetching UMLS data",
    max_workers: int = 10
):
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(fetch_func, cui, apikey): cui
            for cui in cui_list
        }
        with tqdm(total=len(cui_list), desc=desc) as pbar:
            for future in concurrent.futures.as_completed(futures):
                cui = futures[future]
                try:
                    result = future.result()
                    if isinstance(result, list):
                        results.extend(result)
                    else:
                        results.append(result)
                    pbar.set_postfix({"Current CUI": cui})
                except Exception as e:
                    print(f"Error fetching CUI {cui}: {e}")
                pbar.update(1)
    return results

In [ ]:
# Retrieve all atoms for UMLS vaccines (C0042210, C5399710)
#https://documentation.uts.nlm.nih.gov/rest/atoms/
@rate_limited(rate_limiter)
def get_umls_atoms(cui: str, apiKey : str, lang = "ENG", pageNumber = 1 ) -> List[dict]:
    all_atoms = []
    UMLS_ATOM_URL = f'{uri}/content/current/CUI/{{cui}}/atoms'
    while True: 
        response = requests.get(UMLS_ATOM_URL.format(cui=cui), params={'apiKey': apiKey, 'language': lang, 'pageNumber': pageNumber})
        response.encoding = 'utf-8'
        if response.status_code != 200:
            break
        atoms = response.json().get('result', {})
        if not atoms:
            break
        all_atoms.extend(atoms)
        pageNumber += 1
    return all_atoms

In [7]:
# Retrieve all descendants for a given UMLS Atom
# https://documentation.uts.nlm.nih.gov/rest/atoms/ancestors-and-descendants/index.html
@rate_limited(rate_limiter)
def get_umls_descendants(aui: str, apiKey: str, lang: str = "ENG", pageNumber: int = 1) -> List[dict]:
    all_descendants = []
    UMLS_DESCENDANT_URL = f'{uri}/content/current/AUI/{{aui}}/descendants'
    while True:
        response = requests.get(UMLS_DESCENDANT_URL.format(aui=aui), params={'apiKey': apiKey, 'language': lang, 'pageNumber': pageNumber})
        response.encoding = 'utf-8'
        if response.status_code != 200:
            break
        descendants = response.json().get('result', {})
        if not descendants:
            break
        all_descendants.extend(descendants)
        pageNumber += 1
    return all_descendants

In [8]:
# Retrieve all related atoms for a given UMLS concept
# https://documentation.uts.nlm.nih.gov/rest/relations/
@rate_limited(rate_limiter)
def get_related_concepts(cui: str, apiKey: str, relation_types: List[str] = None, pageNumber: int = 1) -> List[dict]:
    all_related = []
    UMLS_RELATION_URL = f'{uri}/content/current/CUI/{{cui}}/relations'
    while True:
        response = requests.get(UMLS_RELATION_URL.format(cui=cui), params={'apiKey': apiKey, 'pageNumber': pageNumber})
        response.encoding = 'utf-8'
        if response.status_code != 200:
            break
        related = response.json().get('result', {})
        if relation_types:
            related = [item for item in related if item.get('relationLabel') in relation_types]
        if not related:
            break
        all_related.extend(related)
        pageNumber += 1
    return all_related

In [9]:
# Retrieve all related atoms for a given atom
#https://documentation.uts.nlm.nih.gov/rest/source-asserted-identifiers/relations/
@rate_limited(rate_limiter)
def get_related_atoms(aui: str, apiKey: str, relations_url:str = None, relation_types: List[str] = None, pageNumber: int = 1) -> List[dict]:
    if not relations_url:
        relations_url = f'{uri}/content/current/AUI/{{aui}}/relations' # needs to be updated with the correct URL
    all_related = []
    while True:
        if relation_types:
            params = {'apiKey': apiKey, 'pageNumber': pageNumber, 'includeRelationLabels': relation_types}
        else:
            params = {'apiKey': apiKey, 'pageNumber': pageNumber}
        response = requests.get(relations_url.format(aui=aui), params=params)
        response.encoding = 'utf-8'
        if response.status_code != 200:
            break
        related = response.json().get('result', {})
        if not related:
            break
        all_related.extend(related)
        pageNumber += 1
    return all_related


In [17]:
# Get umls CUI for atom (https://documentation.uts.nlm.nih.gov/rest/atoms/)
@rate_limited(rate_limiter)
def get_umls_cui(aui: str, apiKey: str, aui_url: str = None) -> Optional[str]:
    if not aui_url:
        aui_url = f'{uri}/content/current/AUI/{{aui}}'
    response = requests.get(aui_url.format(aui=aui), params={'apiKey': apiKey})
    response.encoding = 'utf-8'
    if response.status_code != 200:
        return None
    atom_info = response.json().get('result', {})
    return atom_info.get('concept', None)

In [52]:
# Get rxnorm aui 
@rate_limited(rate_limiter)
def get_rxnorm_aui(rxcui: str, apiKey: str, rxcui_url: str = None) -> Optional[str]:
    if not rxcui_url:
        rxcui_url = f'{uri}/content/current/source/RXNORM/{rxcui}'
    response = requests.get(rxcui_url.format(rxcui=rxcui), params={'apiKey': apiKey})
    response.encoding = 'utf-8'
    if response.status_code != 200:
        return None
    rxcui_info = response.json().get('result', {})
    return rxcui_info.get('ui', None)

#### 1. Vaccine Atoms belonging to 'C0042210', 'C5399710' : 

In [10]:
vaccine_cuis = ['C0042210', 'C5399710']
vaccine_atoms = threaded_umls_fetch(vaccine_cuis, apikey, get_umls_atoms, desc="Fetching UMLS Atoms", max_workers=5)

Fetching UMLS Atoms: 100%|██████████| 2/2 [00:00<00:00,  3.08it/s, Current CUI=C0042210]


In [11]:
vac_atom_df = pd.DataFrame(vaccine_atoms)
vac_atom_df

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents
0,Atom,A32340275,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MTH,PN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine [APC],NONE,NONE,NONE,NONE,NONE,NONE
1,Atom,A32340089,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine,NONE,NONE,NONE,NONE,NONE,NONE
2,Atom,A32340077,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,FN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine [APC],NONE,NONE,NONE,NONE,NONE,NONE
3,Atom,A24665667,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,HC,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccines,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
4,Atom,A22722839,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,ATC,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,VACCINES,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
5,Atom,A32282011,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,NONE
6,Atom,A0131113,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,LCH,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccines,NONE,NONE,NONE,NONE,NONE,NONE
7,Atom,A23870698,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,LCH_NW,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccines,NONE,NONE,NONE,NONE,NONE,NONE
8,Atom,A1397511,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,AOD,DE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,vaccines,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
9,Atom,A0806394,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,SNMI,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Therapeutic vaccine, NOS",https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...


#### 2. Descendant Atoms belonging to all Vaccine Atoms : 

In [ ]:
# Retrieve all source asserted descendents for all atoms (AUIs) of the two Vaccine concepts
atoms = vac_atom_df['ui'].unique().tolist()
atom_descendants = threaded_umls_fetch(atoms, apikey, get_umls_descendants, desc="Fetching UMLS Descendants", max_workers=5)
print(f"{len(atom_descendants)} source asserted descendants found for {len(atoms)} atoms.")

In [ ]:
atom_descendants_df = pd.DataFrame(atom_descendants)
atom_descendants_df.to_pickle('results/01_atom_descendants.pkl')

In [12]:
atom_descendants_df = pd.read_pickle('results/01_atom_descendants.pkl')
atom_descendants_df

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents
0,Atom,A0320257,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,AOD,DE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,viral vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
1,Atom,A0318079,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,AOD,DE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,AIDS vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
2,Atom,A0474344,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,AOD,DE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,bacterial vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
3,Atom,A28934538,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Inactivated Pneumococcal Vaccine,NONE,NONE,NONE,NONE,NONE,NONE
4,Atom,A28934573,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Live Attenuated Rotavirus Vaccine,NONE,NONE,NONE,NONE,NONE,NONE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3313,Atom,A21401058,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,NCI,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Autologous Pluripotent ALDHbr Stem Cells ALD-451,NONE,NONE,NONE,NONE,NONE,NONE
3314,Atom,A23885829,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,NCI,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,IL-12-expressing Mesenchymal Stem Cell Vaccine...,NONE,NONE,NONE,NONE,NONE,NONE
3315,Atom,A29762108,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,NCI,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Tonogenchoncel-L,NONE,NONE,NONE,NONE,NONE,NONE
3316,Atom,A32403905,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,NCI,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Allogeneic Glial Progenitor Cells,NONE,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE


#### 3. Atoms related to vaccine atoms: 

In [ ]:
# Retrieve all broader concepts (similar to children concepts) for all atoms (AUIs) of the two Vaccine concepts - This provides atoms outside of source vocabularies
# get all values from 'relation' column that is not NONE
atom_rela = vac_atom_df['relations'].unique().tolist()
atom_rela = [item for item in atom_rela if item != 'NONE']
atoms_relations = []
# NEED TO extract just the atoms from the relations. 
for atom_rel in atom_rela: 
    atoms_relations.extend(get_related_atoms(atom_rel, apikey, atom_rel, ["RB", "RO", "RQ", "SY", "RL"]))


In [ ]:
related_df = pd.DataFrame(atoms_relations)
# remove records where column 'additionalRelationLabel' is 'has_translation'
related_df = related_df[related_df['additionalRelationLabel'] != 'has_translation']

In [14]:
related_df['codeClean'] = related_df['relatedId'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)

In [16]:
#Major ToDo: Iterate through descendants key till there is NONE [p2]

In [20]:
# Get source concept (https://documentation.uts.nlm.nih.gov/rest/atoms/)
related_df['sourceCUI'] = related_df.apply(lambda x: get_umls_cui(x['ui'], apikey, x['relatedId']), axis=1)

In [21]:
related_df

,ui,sourceUi,obsolete,suppressible,sourceOriginated,rootSource,relationLabel,additionalRelationLabel,groupId,relatedId,relatedIdName,relatedFromId,relatedFromIdName,attributeCount,classType,codeClean,sourceCUI
0,R00690846,NONE,False,False,False,AOD,RB,,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,viral vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,vaccines,0,AtomRelation,A0320257,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
1,R00691155,NONE,False,False,False,AOD,RB,,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,bacterial vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,vaccines,0,AtomRelation,A0474344,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
2,R123774609,NONE,False,False,False,VANDF,RB,inverse_isa,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MENINGOCOCCAL OLIGOSACCHARIDE CONJUGATE VACCIN...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,VACCINES,0,AtomRelation,A18803744,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
3,R211439176,NONE,False,False,False,VANDF,RB,inverse_isa,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MENINGOCOCCAL OLIGOSACCHAR CONJ VACCINE (MENVE...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,VACCINES,0,AtomRelation,A34967388,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
4,R70583324,NONE,False,False,False,VANDF,RB,inverse_isa,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,HEPATITIS A 720 EL.U/HEPATITIS B 20MCG/ML VACC...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,VACCINES,0,AtomRelation,A8451724,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,R148149150,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,SNC-rCTB vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C583182,None
263,R148181673,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,TA-NIC,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C582229,None
264,R148235829,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NicVAX,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C557119,None
265,R148154736,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Sj22.6-26GST vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C573432,None


In [22]:
related_df.to_pickle('results/02_related_df.pkl')

In [23]:
related_df = pd.read_pickle('results/02_related_df.pkl')
related_df

,ui,sourceUi,obsolete,suppressible,sourceOriginated,rootSource,relationLabel,additionalRelationLabel,groupId,relatedId,relatedIdName,relatedFromId,relatedFromIdName,attributeCount,classType,codeClean,sourceCUI
0,R00690846,NONE,False,False,False,AOD,RB,,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,viral vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,vaccines,0,AtomRelation,A0320257,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
1,R00691155,NONE,False,False,False,AOD,RB,,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,bacterial vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,vaccines,0,AtomRelation,A0474344,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
2,R123774609,NONE,False,False,False,VANDF,RB,inverse_isa,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MENINGOCOCCAL OLIGOSACCHARIDE CONJUGATE VACCIN...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,VACCINES,0,AtomRelation,A18803744,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
3,R211439176,NONE,False,False,False,VANDF,RB,inverse_isa,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MENINGOCOCCAL OLIGOSACCHAR CONJ VACCINE (MENVE...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,VACCINES,0,AtomRelation,A34967388,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
4,R70583324,NONE,False,False,False,VANDF,RB,inverse_isa,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,HEPATITIS A 720 EL.U/HEPATITIS B 20MCG/ML VACC...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,VACCINES,0,AtomRelation,A8451724,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
262,R148149150,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,SNC-rCTB vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C583182,None
263,R148181673,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,TA-NIC,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C582229,None
264,R148235829,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NicVAX,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C557119,None
265,R148154736,NONE,False,False,False,MSH,RB,mapped_from,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Sj22.6-26GST vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,Vaccines,0,AtomClusterRelation,C573432,None


#### 3.5. All atoms of concepts found so far

In [29]:
related_df_cui_list = related_df['sourceCUI'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x).dropna().unique().tolist()
atoms_of_related_df_cui_list = threaded_umls_fetch(related_df_cui_list, apikey, get_umls_atoms, desc="Fetching Atoms of Related CUI List", max_workers=5)

Fetching Atoms of Related CUI List: 100%|██████████| 205/205 [00:22<00:00,  9.11it/s, Current CUI=C0042213]


In [33]:
related_atom_df = pd.DataFrame(atoms_of_related_df_cui_list)
related_atom_df.to_pickle('results/03_related_atom_df.pkl')

In [34]:
related_atom_df = pd.read_pickle('results/03_related_atom_df.pkl')
related_atom_df

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents
0,Atom,A0132653,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MTH,PN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Viral Vaccines,NONE,NONE,NONE,NONE,NONE,NONE
1,Atom,A18576064,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,CHV,SY,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,viral vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,NONE
2,Atom,A0320258,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,CSP,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,viral vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
3,Atom,A0470799,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,LCH,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Viral vaccines,NONE,NONE,NONE,NONE,NONE,NONE
4,Atom,A15199010,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MEDCIN,FN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,viral vaccines (medication),NONE,NONE,NONE,NONE,NONE,NONE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1713,Atom,A32284447,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,PM,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Vaccine, Synthetic",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE
1714,Atom,A26604469,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Synthetic Immunogens,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE
1715,Atom,A0131129,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,MH,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Vaccines, Synthetic",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE
1716,Atom,A26644974,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Synthetic Vaccines,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE


In [35]:
# Check the total unique atoms
vac_atom_df_atoms = vac_atom_df['ui'].unique().tolist()
atom_descendants_df_atoms = atom_descendants_df['ui'].unique().tolist()
related_atom_df_atoms = related_atom_df['ui'].unique().tolist()
atom_list = vac_atom_df_atoms + atom_descendants_df_atoms + related_atom_df_atoms
total_unique_atoms = set()
for atom in atom_list:
    if isinstance(atom, str):
        total_unique_atoms.add(atom)
len(total_unique_atoms)

4242

#### 3.5.1 Bandage for Rxnorm: 

In [50]:
rxnav_df = pd.read_pickle("rxnav_data.pkl")
rxnav_df

,rxcui,label,status,tty
0,2566422,0.5 ML Streptococcus pneumoniae serotype 1 cap...,Active,SBD
1,2687724,Fluzone 2024-2025,Active,BN
2,1601400,Neisseria meningitidis serogroup B strain NZ98...,Active,SCDC
3,2270606,Vibrio cholerae CVD 103-HGR strain live antige...,Active,SCDG
4,1190919,"0.5 ML diphtheria toxoid vaccine, inactivated ...",Active,SBD
...,...,...,...,...
1879,2688113,influenza A virus A/Sydney/1304/2022 (H3N2) an...,Active,PIN
1880,2685010,Mresvia Injectable Product,Active,SBDG
1881,2270605,Vibrio cholerae CVD 103-HGR strain live antige...,Active,SCDG
1882,798231,Streptococcus pneumoniae serotype 6B capsular ...,Active,SCDC


In [54]:
rxcui_list = rxnav_df['rxcui'].dropna().unique().tolist()
rxcui_auis = threaded_umls_fetch(rxcui_list, apikey, get_rxnorm_aui, desc="Fetching RxNorm AUIs", max_workers=15)

Fetching RxNorm AUIs:   0%|          | 0/1884 [00:00<?, ?it/s]

Fetching RxNorm AUIs: 100%|██████████| 1884/1884 [04:11<00:00,  7.48it/s, Current CUI=1657429]


In [64]:
rxcui_auis

['2566422',
 '2687724',
 '1601400',
 '2270606',
 '1190919',
 '1597086',
 '2566408',
 '2642140',
 '1300369',
 '2694019',
 '2593120',
 '1657137',
 '2583643',
 '2280749',
 '2561244',
 '2588240',
 '2603506',
 '2170518',
 '2197292',
 '2109619',
 '2566419',
 '2648742',
 '2664840',
 '2660432',
 '2642409',
 '2645589',
 '798298',
 '1160325',
 '2642408',
 '1994348',
 '1300190',
 '2661479',
 '2270607',
 '2645752',
 '901632',
 '2642412',
 '2649176',
 '1666421',
 '94317',
 '901643',
 None,
 '2648898',
 '2566301',
 '2583644',
 '2641851',
 '798276',
 '2050419',
 '1300307',
 '1653483',
 '2691207',
 '798226',
 '901641',
 '1292441',
 '2605553',
 '854938',
 '845527',
 '1657222',
 '2661458',
 '2677448',
 '2606078',
 '2646500',
 '2664742',
 '830549',
 '2686922',
 '2468230',
 '2685003',
 '2360576',
 '2648632',
 '2603832',
 '854951',
 '1658035',
 '1300384',
 '2636592',
 '2109617',
 '2664827',
 '2624749',
 '2687362',
 '2646219',
 '830261',
 '1597083',
 '798222',
 '901506',
 '2642311',
 None,
 '2658504',
 '262

In [59]:
rxnorm_cui = threaded_umls_fetch(rxcui_auis, apikey, get_umls_cui, desc="Fetching RxNorm CUI", max_workers=15)

Fetching RxNorm CUI: 100%|██████████| 1884/1884 [03:08<00:00,  9.99it/s, Current CUI=1657429]


In [62]:
rxnorm_cui_df = pd.DataFrame(rxnorm_cui)
rxnorm_cui_df

,0
0,None
1,None
2,None
3,None
4,None
...,...
1879,None
1880,None
1881,None
1882,None


In [38]:
concatonated_df = pd.concat([vac_atom_df, atom_descendants_df, related_atom_df], ignore_index=True)
concatonated_df

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents
0,Atom,A32340275,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MTH,PN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine [APC],NONE,NONE,NONE,NONE,NONE,NONE
1,Atom,A32340089,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine,NONE,NONE,NONE,NONE,NONE,NONE
2,Atom,A32340077,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,FN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine [APC],NONE,NONE,NONE,NONE,NONE,NONE
3,Atom,A24665667,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,HC,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccines,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
4,Atom,A22722839,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,ATC,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,VACCINES,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5067,Atom,A32284447,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,PM,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Vaccine, Synthetic",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE
5068,Atom,A26604469,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Synthetic Immunogens,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE
5069,Atom,A0131129,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,MH,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Vaccines, Synthetic",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE
5070,Atom,A26644974,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Synthetic Vaccines,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE


In [40]:
concatonated_df['rootSource'].value_counts()

rootSource
NCI              1531
MEDCIN            684
SNOMEDCT_US       532
RCD               348
RXNORM            304
MSH               294
NDDF              223
VANDF             152
ATC               135
CHV                81
CSP                73
SNMI               71
CVX                56
MTH                54
MMX                48
MMSL               47
GS                 39
USPMG              35
MEDLINEPLUS        33
MED-RT             32
MDR                32
LNC                25
MTHSPL             24
AOD                21
ICPC2ICD10ENG      20
SNM                19
ICPC2P             17
PDQ                14
LCH_NW             14
LCH                11
PSY                 9
ICD10CM             8
CST                 7
DXP                 7
HL7V3.0             6
COSTAR              6
CCS                 6
ICD10               5
BI                  5
NANDA-I             5
ICD10AM             5
CCPSS               4
OMIM                4
CCSR_ICD10CM        3
MTHICD9             3

#### 4. Related Atoms belonging to all vaccine descendant atoms: 

In [ ]:
#load umls_vaccine_descendants into a pandas dataframe
vac_df = pd.DataFrame(atom_descendants)
vac_df.head()
vac_df.to_pickle('results/umls_vaccine_descendants.pkl')

In [ ]:
vac_df = pd.read_pickle('results/umls_vaccine_descendants.pkl')

In [ ]:
# new column: codeClean where all string after the last '/' is extracted from the code column
vac_df['codeClean'] = vac_df['code'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
vac_df['conceptClean'] = vac_df['concept'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
# ToDo: There are some concepts which may not be appropriate (sematic types: Findings, Therapeutic or Preventive Procedure, Health Care Activity etc.) May need to remove them
vac_df

In [ ]:
atoms = vac_df['ui'].unique().tolist()
print(f"Total unique atoms: {len(atoms)}")
related_concepts = threaded_umls_fetch(atoms, apikey, get_umls_descendants, desc="Fetching UMLS Descendants", max_workers=5)

In [ ]:
len(related_concepts)

In [ ]:
related_df = pd.DataFrame(related_concepts)
related_df

In [ ]:
related_df['codeClean'] = related_df['relatedId'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)

In [ ]:
related_df[related_df['codeClean'] == 'A36802459']

#### Test Case: Missing Concept A36802459 - C5941869
> 0.3 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT162b2 OMICRON (KP.2) 0.1 MG/ML Prefilled Syringe [Comirnaty 2024-2025]

In [ ]:
# check if 'A36802459' is in 'ui' column 
vac_df[vac_df['ui'] == 'A36802459']

### Workaround: Try to explode concept column to get atoms again. 

In [ ]:
cui_list = vac_df['conceptClean'].unique().tolist()

In [ ]:
len(cui_list)

In [ ]:
exploded_atoms = get_all_atoms_threaded(cui_list, apikey, max_workers=10)

In [ ]:
len(exploded_atoms)

In [ ]:
exploded_df = pd.DataFrame(exploded_atoms)
exploded_df.head()
exploded_df.to_pickle('results/umls_exploded_atoms.pkl')

In [ ]:
exploded_df = pd.read_pickle('results/umls_exploded_atoms.pkl')

In [ ]:
exploded_df['codeClean'] = exploded_df['code'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
exploded_df['conceptClean'] = exploded_df['concept'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
exploded_df

In [ ]:
exploded_df['rootSource'].value_counts()

In [ ]:
exploded_df[exploded_df['ui'] == 'A36802459']

In [ ]:
exploded_df[exploded_df['rootSource'] == 'RXNORM']

In [ ]:
exploded_df[exploded_df['rootSource'] == 'MSH']


In [ ]:
# concatonate exploded_df and vac_df 
combined_df = pd.concat([exploded_df, vac_df], ignore_index=True)
# remove duplicates based on 'ui' column
combined_df = combined_df.drop_duplicates(subset='ui', keep='first')
combined_df['rootSource'].value_counts()

In [ ]:
# Unique number of conceptClean column
combined_df['conceptClean'].nunique()

In [ ]:
combined_df

In [45]:
athena_to_umls_map = {
    "MeSH": "MSH",                  # Medical Subject Headings
    "RxNorm Extension": "RXNORM_EXT",   # Usually merged into RXNORM in UMLS
    "RxNorm": "RXNORM",
    "GCN_SEQNO": "NDDF",             # Generic Code Number from First DataBank
    "SPL": "MTHSPL",                 # Structured Product Labeling
    "NDC": "NDC",                    # National Drug Code (if present in UMLS)
    "SNOMED": "SNOMEDCT_US",
    "CVX": "CVX",
    "CPT4": "CPT",                   
    "VANDF": "VANDF",
    "ATC": "ATC",
    "EphMRA ATC": "ATC",             # Map to ATC
    "HCPCS": "HCPCS",                
    "SNOMED Veterinary": "SNOMEDCT_VET", 
    "NDFRT": "VANDF",               
    "NCCD": "NCCD",                  
    "OMOP Invest Drug": "OMOPID",    
    "ICD10PCS": "ICD10PCS",          
    "GGR": "GGR",                    
    "KDC": "KDC",                    # Same
    "AMT": "AMT",                    # Australian Medicines Terminology
    "Nebraska Lexicon": "NEB_LEX", # No mapping in UMLS list
    "CO-CONNECT": "CO-CONNECT",      # Same
    "VA Class": "VANDF",             
    "dm+d": "DM+D",                  # UK dictionary of medicines + devices
    "HemOnc": "HemOnc",              # No UMLS equivalent
    "JMDC": "JMDC",                  # Same
    "BDPM": "BDPM",
    "CTD": "CTD",
    "EDI": "EDI",
    "DPD": "DPD",
    "Multum": "MMSL",                # Multum MedIndex = MMSL in UMLS
    "Read": "RCD",                   # Read Codes
    "CIEL": "CIEL"                   # Columbia International eHealth Lab
}


In [46]:
# read csv into dataframe
athena_df = pd.read_csv('results/Athena search[vaccine_valid_drug].csv', sep='\t')
athena_df.head()

,Id,Code,Name,Standard Class,Domain,Vocab,Validity,Concept
0,28493,D000068756,Valsartan,Main Heading,Drug,MeSH,Valid,Non-standard
1,28686,C576297,ponceau 4R,Suppl Concept,Drug,MeSH,Valid,Non-standard
2,32766,OMOP4873977,Middle East Respiratory Syndrome heterologous ...,Ingredient,Drug,RxNorm Extension,Valid,Standard
3,509104,347699,meningococcal group A polysaccharide 0.1 MG/ML...,Clinical Drug,Drug,RxNorm,Valid,Standard
4,514012,314724,meningococcal polysaccharide vaccine group W-135,Ingredient,Drug,RxNorm,Valid,Standard


In [65]:
athena_df['Vocab'].value_counts()

Vocab
RxNorm Extension     9156
RxNorm               2020
SPL                  1759
NDC                   862
dm+d                  840
AMT                   672
CPT4                  468
NDFRT                 333
SNOMED Veterinary     311
GCN_SEQNO             280
Nebraska Lexicon      255
SNOMED                250
MeSH                  202
CVX                   198
VANDF                 162
OMOP Invest Drug      153
BDPM                   88
DPD                    81
CIEL                   70
JMDC                   57
ATC                    52
Multum                 45
NCCD                   23
HCPCS                  21
EphMRA ATC             18
KDC                    13
CTD                    13
ICD10PCS               12
Read                   11
CO-CONNECT              8
HemOnc                  4
VA Class                3
GGR                     1
EDI                     1
Name: count, dtype: int64

In [47]:
athena_df['Vocab_UMLS'] = athena_df['Vocab'].map(athena_to_umls_map)

In [66]:
concatonated_df['rootSource'].value_counts()

rootSource
NCI              1531
MEDCIN            684
SNOMEDCT_US       532
RCD               348
RXNORM            304
MSH               294
NDDF              223
VANDF             152
ATC               135
CHV                81
CSP                73
SNMI               71
CVX                56
MTH                54
MMX                48
MMSL               47
GS                 39
USPMG              35
MEDLINEPLUS        33
MED-RT             32
MDR                32
LNC                25
MTHSPL             24
AOD                21
ICPC2ICD10ENG      20
SNM                19
ICPC2P             17
PDQ                14
LCH_NW             14
LCH                11
PSY                 9
ICD10CM             8
CST                 7
DXP                 7
HL7V3.0             6
COSTAR              6
CCS                 6
ICD10               5
BI                  5
NANDA-I             5
ICD10AM             5
CCPSS               4
OMIM                4
CCSR_ICD10CM        3
MTHICD9             3

In [48]:
# show the valuecounts of the 'Vocab' column (athena_df) and 'rootSource' column (combined_df) side by side
vocab_counts = athena_df['Vocab_UMLS'].value_counts()
# root_source_counts = combined_df['rootSource'].value_counts()
root_source_counts = concatonated_df['rootSource'].value_counts()
comparison_df = pd.DataFrame({'Athena': vocab_counts,
'UMLS': root_source_counts}).fillna(0).astype(int)
comparison_df

,Athena,UMLS
AMT,672,0
AOD,0,21
ATC,70,135
BDPM,88,0
BI,0,5
...,...,...
SNOMEDCT_US,250,532
SNOMEDCT_VET,311,0
USPMG,0,35
VANDF,498,152


In [ ]:
comparison_df

### ToDo: 
1. Add function to get related concepts (https://documentation.uts.nlm.nih.gov/rest/relations/)
2. Normalize vocab sources acros Athena and UMLS
3. Check vocabulary code to identify missing concepts from each source